# Example 1

In [1]:
import rapidsegment as rs
from prettytable import PrettyTable
import pandas as pd
from rapidsegment import UniversalDataLoader
from rapidsegment import StrategicSegmentBuilder
from rapidsegment import StrategicSegmentScore
import duckdb
import pyarrow as pa
import os
pd.options.display.max_columns = 100

In [2]:
print(f"RapidSegment version: {rs.__version__}")

RapidSegment version: 1.2.9.post1


# LOAD DATA

In [3]:
data = UniversalDataLoader(file_path=r"/teamspace/studios/this_studio/RapidSegment/Notebooks/Adult_Income_Dataset/adult.csv", ).load()
print(f"Loaded as {type(data)} table for better performance ")
data.slice(0,5).to_pandas()

2026-09-05 08:22:08,486 | INFO     | [data_loader.py:148] | 📂 Loading file: /teamspace/studios/this_studio/RapidSegment/Notebooks/Adult_Income_Dataset/adult.csv (extension: .csv)


Loaded as <class 'pyarrow.lib.Table'> table for better performance 


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90.0,?,77053.0,HS-grad,9.0,Widowed,?,Not-in-family,White,Female,0.0,4356.0,40.0,United-States,<=50K
1,82.0,Private,132870.0,HS-grad,9.0,Widowed,Exec-managerial,Not-in-family,White,Female,0.0,4356.0,18.0,United-States,<=50K
2,66.0,?,186061.0,Some-college,10.0,Widowed,?,Unmarried,Black,Female,0.0,4356.0,40.0,United-States,<=50K
3,54.0,Private,140359.0,7th-8th,4.0,Divorced,Machine-op-inspct,Unmarried,White,Female,0.0,3900.0,40.0,United-States,<=50K
4,41.0,Private,264663.0,Some-college,10.0,Separated,Prof-specialty,Own-child,White,Female,0.0,3900.0,40.0,United-States,<=50K


#### Notes
- Here income is the target variable. But we see that it is not one hot encoded or not binary in nature.
- We need to binarize the variable before we proceed to segment creation

# Binarizing Target Variable
    Using duckdb SQL native syntax to perform the binarization

In [4]:
# specify a database file path for persistence
db_file = "dataset.duckdb"

# connect to the duckdb database file (creates it if it doesn't exist)
# read_only=False is important for writing/modifying data
con = duckdb.connect(database=db_file, read_only=False)

# register the pyarrow table as a virtual table named 'original_data'
# duckdb can query this table as if it were a native table in the db
duckdb_data_var = "original_data"
con.register(duckdb_data_var, data)

print(f"\nduckdb database connected and persisted to: {db_file}")
print(f"pyarrow table registered as {duckdb_data_var}")


duckdb database connected and persisted to: dataset.duckdb
pyarrow table registered as original_data


In [5]:
# define the sql query
# this converts 'active' status to 1, and all other statuses to 0
target_col = "income"
sql_query = f"""
SELECT
    *,
    CASE WHEN {target_col} = '>50K' THEN 1 ELSE 0 END AS {target_col}_binary
FROM
    {duckdb_data_var}
"""

# execute the query and fetch the result directly as a pyarrow table
mod_data = con.execute(sql_query).arrow()
mod_data = pa.Table.from_batches(mod_data)
print("\nduckdb query executed. result as pyarrow table:")
mod_data.slice(0,5).to_pandas()


duckdb query executed. result as pyarrow table:


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income,income_binary
0,90.0,?,77053.0,HS-grad,9.0,Widowed,?,Not-in-family,White,Female,0.0,4356.0,40.0,United-States,<=50K,0
1,82.0,Private,132870.0,HS-grad,9.0,Widowed,Exec-managerial,Not-in-family,White,Female,0.0,4356.0,18.0,United-States,<=50K,0
2,66.0,?,186061.0,Some-college,10.0,Widowed,?,Unmarried,Black,Female,0.0,4356.0,40.0,United-States,<=50K,0
3,54.0,Private,140359.0,7th-8th,4.0,Divorced,Machine-op-inspct,Unmarried,White,Female,0.0,3900.0,40.0,United-States,<=50K,0
4,41.0,Private,264663.0,Some-college,10.0,Separated,Prof-specialty,Own-child,White,Female,0.0,3900.0,40.0,United-States,<=50K,0


# Setup Experiment

Naive path


In [6]:
param_grid = {'min_sample_size': [20000, 15000, 10000, 5000, 500],'min_lift': [3.0, 2.0, 1.5]}
builder = StrategicSegmentBuilder(target = target_col+'_binary',
                                  min_sample_size = 100,
                                  min_lift = 1.0, 
                                  min_events  = 50, 
                                  top_n_vars = 10,
                                  max_segments = 10,
                                  param_grid = param_grid,
                                  enable_diversity = False, 
                                  max_feature_reuse = 5, 
                                  enable_1way = True,
                                  enable_2way = True,
                                  enable_3way = True,
                                  feature_groups = None,
                                  ignore_features = [target_col],
                                  sort_priority = 'lift_rate_count',
                                  binning_method="naive",
                                  selection_metric='response_rate',
                                  expand_log_mode='champion',
                                  max_expansion_hops=0,)

#### Rule Mining

In [7]:
segments_df = builder.extract_segments(mod_data)

2026-09-05 08:22:08,644 | INFO     | [builder.py:1228] | 🚀 Starting hierarchical segment extraction...
2026-09-05 08:22:08,646 | INFO     | [builder.py:1253] | 📂 Created temporary disk-backed DB at: experiments/segmentation_20260905_12444aa0.duckdb
2026-09-05 08:22:08,675 | INFO     | [builder.py:1277] | ⚙️ DuckDB Configured for Disk Spilling: Threads=4/4, MemoryLimit=11GB, TempDir=None
2026-09-05 08:22:08,675 | INFO     | [builder.py:1281] | 📊 Sort priority: lift_rate_count
2026-09-05 08:22:08,676 | INFO     | [builder.py:1282] | 📦 Binning method: naive (naive_bins=5)
2026-09-05 08:22:08,783 | INFO     | [builder.py:1365] | 📊 Dynamic Grid Search Enabled: 15 configurations.
2026-09-05 08:22:08,786 | INFO     | [builder.py:1373] | 🔒 Locking Original Base Rate: 24.08%
2026-09-05 08:22:08,788 | INFO     | [builder.py:1399] | 🔄 Iteration 1 | Remaining Volume: 32,561 | Base Rate: 24.08%
2026-09-05 08:22:08,789 | INFO     | [builder.py:340] | 🔍 Computing IV and bins for 14 features...
2026-0

### Segment Evaluation

In [8]:
final_eval = builder.evaluate_final_coverage(mod_data)

2026-09-05 08:22:24,917 | INFO     | [builder.py:1810] | 📊 Evaluating final hierarchical coverage on original data...


# Final Segment Report

In [9]:

table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------------------+--------------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |    capture_rate    |        lift        | cumulative_sample_capture | cumulative_event_capture |
+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------------------+--------------------------+
|   1.0   |    1160.0   |     950.0     | 81.89655172413794  | 24.080955744602438 | 3.5625441479070052 | 3.4008846074348367 |     3.5625441479070052    |    12.115801555923989    |
|   2.0   |    554.0    |     424.0     | 76.53429602888086  | 24.080955744602438 | 1.7014219465004146 | 3.1782084083616753 |      5.26396609440742     |    17.523275092462697    |
|   3.0   |    505.0    |     375.0     | 74.25742574257426  | 24.080955744602438 | 1.550935167

# Segment SQL definition

In [10]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   education.num=[13.0, inf) & occupation=[Exec-managerial] & relationship=[Husband]
SQL Filter: ("education.num" >= 13.0) AND ("occupation" IN ('Exec-managerial')) AND ("relationship" IN ('Husband'))
--------------------------------------------------
Segment ID: 2
Raw Rule:   age=[41.0, 50.0) & occupation=[Prof-specialty] & relationship=[Husband]
SQL Filter: ("age" >= 41.0 AND "age" < 50.0) AND ("occupation" IN ('Prof-specialty')) AND ("relationship" IN ('Husband'))
--------------------------------------------------
Segment ID: 3
Raw Rule:   age=[33.0, 40.0) & marital.status=[Married-civ-spouse] & occupation=[Prof-specialty]
SQL Filter: ("age" >= 33.0 AND "age" < 40.0) AND ("marital.status" IN ('Married-civ-spouse')) AND ("occupation" IN ('Prof-specialty'))
--------------------------------------------------
Segment ID: 4
Raw Rule:   education.num=[12.0, inf) & marital.status=[Married-civ-spouse] & occupation=[Prof-specialty]
SQL Filte

##### Segment Meta informations
-  We can see the raw string and the parsed sql here in the below table.
- We can also see what hyper parameters were satisfied to mine the segment

In [11]:

table = PrettyTable()
table.field_names = list(pd.DataFrame(segments_df).columns)
for _, row in pd.DataFrame(segments_df).iterrows():
    table.add_row(list(row))
print(table)

+------------+-----------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------+-------+--------------------+--------------------+--------------------------+-----------------------+
| segment_id |                                          rule_string                                          |                                                         sql_filter                                                         | count |        rate        |        lift        | meta_applied_sample_size | meta_applied_min_lift |
+------------+-----------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------+-------+--------------------+--------------------+--------------------------+--------------

# Diagnostics

- Check why no futher segments were generated

In [12]:
print(builder.explain_no_segments())

SEGMENT EXTRACTION DIAGNOSTIC REPORT
Segments Extracted : 10 / 10
Iterations Run     : 10
Stop Reason        : Reached max_segments limit (10).
--------------------------------------------------------------------------------
Active Constraints:
  - min_sample_size : 100
  - min_lift        : 1.00x
  - min_events      : 50
  - selection_metric: response_rate

Feature Eligibility Summary (Iteration 10):
  - 6   feature(s): Eligible for Combination Search
  - 4   feature(s): Excluded (Max Feature Reuse Exceeded)
  - 4   feature(s): Excluded (Outside Top N Features by Score)

Candidate Funnel (Iteration 10):
  - 1-way candidates passing base criteria : 0
  - 2-way candidates passing base criteria : 0
  - 3-way candidates passing base criteria : 0
  - Total candidates before grid search   : 336
  - Candidates clearing grid filter       : 1



##### Check why a feature was selected/not selected
- Here we see that AGE was selected many times
- The selection was due the fact that we allowed feature resue
- And seems age was an imporatnt variable as seen from the dynamic response rate values


In [13]:
builder.explain_feature_journey("age")

📌 AUDIT TRAIL FOR FEATURE: 'age'

[Iteration 1]
  • Current dynamic RESPONSE_RATE   : 0.3752
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : education.num=[13.0, inf) & occupation=[Exec-managerial] & relationship=[Husband] (Variables: ['education.num', 'occupation', 'relationship'])

[Iteration 2]
  • Current dynamic RESPONSE_RATE   : 0.3427
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  🎉 SELECTED as part of winning rule!
     Rule: age=[41.0, 50.0) & occupation=[Prof-specialty] & relationship=[Husband]

[Iteration 3]
  • Current dynamic RESPONSE_RATE   : 0.3087
  • Previous times used  : 1
  • Selection Status     : Eligible for Combination Search
  🎉 SELECTED as part of winning rule!
     Rule: age=[33.0, 40.0) & marital.status=[Married-civ-spouse] & occupation=[Prof-specialty]

[Iteration 4]
  • Current dynamic RESPONSE_RATE   : 0.3096
  • Previous times used  : 2
  • Selec

# Preparing the dataset for scoring and decile banding.
- Only score when atleast 10 segments are found

In [14]:
# define the sql query
# this converts 'active' status to 1, and all other statuses to 0
sql_query = f"""
SELECT *, 
CASE 
    WHEN ("education.num" >= 13.0) AND ("occupation" IN ('Exec-managerial')) AND ("relationship" IN ('Husband')) THEN 1 
    ELSE 0 
END AS seg_1,

CASE 
    WHEN ("age" >= 41.0 AND "age" < 50.0) AND ("occupation" IN ('Prof-specialty')) AND ("relationship" IN ('Husband')) THEN 1 
    ELSE 0 
END AS seg_2,

CASE 
    WHEN ("age" >= 33.0 AND "age" < 40.0) AND ("marital.status" IN ('Married-civ-spouse')) AND ("occupation" IN ('Prof-specialty')) THEN 1 
    ELSE 0 
END AS seg_3,

CASE 
    WHEN ("education.num" >= 12.0) AND ("marital.status" IN ('Married-civ-spouse')) AND ("occupation" IN ('Prof-specialty')) THEN 1 
    ELSE 0 
END AS seg_4,

CASE 
    WHEN ("education.num" >= 12.0) AND ("occupation" IN ('Sales')) AND ("relationship" IN ('Husband')) THEN 1 
    ELSE 0 
END AS seg_5,

CASE 
    WHEN ("age" >= 40.0 AND "age" < 50.0) AND ("education.num" >= 11.0) AND ("marital.status" IN ('Married-civ-spouse')) THEN 1 
    ELSE 0 
END AS seg_6,

CASE 
    WHEN ("age" >= 50.0) AND ("education.num" >= 11.0) AND ("marital.status" IN ('Married-civ-spouse')) THEN 1 
    ELSE 0 
END AS seg_7,

CASE 
    WHEN ("age" >= 39.0 AND "age" < 50.0) AND ("education" IN ('Some-college')) AND ("marital.status" IN ('Married-civ-spouse')) THEN 1 
    ELSE 0 
END AS seg_8,

CASE 
    WHEN ("education" IN ('Some-college')) AND ("hours.per.week" >= 45.0) AND ("relationship" IN ('Husband')) THEN 1 
    ELSE 0 
END AS seg_9,

CASE 
    WHEN ("hours.per.week" >= 40.0 AND "hours.per.week" < 45.0) AND ("relationship" IN ('Wife')) THEN 1 
    ELSE 0 
END AS seg_10,                                                                              
ROW_NUMBER() OVER () AS ID,
FROM mod_data
"""

# execute the query and fetch the result directly as a pyarrow table
pred_data = con.execute(sql_query).arrow()
pred_data = pa.Table.from_batches(pred_data)
print("\nduckdb query executed. result as pyarrow table:")
pred_data.slice(0,5).to_pandas()


duckdb query executed. result as pyarrow table:


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income,income_binary,seg_1,seg_2,seg_3,seg_4,seg_5,seg_6,seg_7,seg_8,seg_9,seg_10,ID
0,90.0,?,77053.0,HS-grad,9.0,Widowed,?,Not-in-family,White,Female,0.0,4356.0,40.0,United-States,<=50K,0,0,0,0,0,0,0,0,0,0,0,1
1,82.0,Private,132870.0,HS-grad,9.0,Widowed,Exec-managerial,Not-in-family,White,Female,0.0,4356.0,18.0,United-States,<=50K,0,0,0,0,0,0,0,0,0,0,0,2
2,66.0,?,186061.0,Some-college,10.0,Widowed,?,Unmarried,Black,Female,0.0,4356.0,40.0,United-States,<=50K,0,0,0,0,0,0,0,0,0,0,0,3
3,54.0,Private,140359.0,7th-8th,4.0,Divorced,Machine-op-inspct,Unmarried,White,Female,0.0,3900.0,40.0,United-States,<=50K,0,0,0,0,0,0,0,0,0,0,0,4
4,41.0,Private,264663.0,Some-college,10.0,Separated,Prof-specialty,Own-child,White,Female,0.0,3900.0,40.0,United-States,<=50K,0,0,0,0,0,0,0,0,0,0,0,5


# Score the segments on the dataset and create decile bands


In [15]:
scorer = StrategicSegmentScore(
    target_col=target_col+'_binary',
    primary_key="ID",
    segment_cols=["seg_1","seg_2",'seg_3','seg_4','seg_5','seg_6','seg_7','seg_8','seg_9','seg_10'],
)

##### Export Segment Score as JSON

In [16]:
model_artifact = scorer.calculate_and_export_weights(pred_data)

2026-09-05 08:22:25,208 | INFO     | [scorer.py:78] | 🚀 Initialising out‑of‑core DuckDB scorecard engine...
2026-09-05 08:22:25,353 | INFO     | [scorer.py:136] | 📊 Computing scorecard weights...
2026-09-05 08:22:25,354 | WARNING  | [scorer.py:180] | ⚠️ DECILE RESOLUTION WARNING: Only 9 distinct non-zero score values found across 10 segments. Splitting into 10 deciles will produce repeated thresholds (e.g., top 5 deciles may have identical scores). For smooth decile ranking, ensure the builder discovers at least 10 distinct segments (increase `max_segments`). Consider interpreting results as tiers rather than deciles.
2026-09-05 08:22:25,355 | INFO     | [scorer.py:194] | ⚡ Scoring population natively via SQL engine...
2026-09-05 08:22:25,371 | INFO     | [scorer.py:214] | 📉 Dataset Zero‑Inflation Rate: 75.92%
2026-09-05 08:22:25,372 | INFO     | [scorer.py:219] | 📈 Calibrating deciles across active populations...
2026-09-05 08:22:25,377 | INFO     | [scorer.py:267] | ✅ Scorecard expor

##### View segment score and create final Decile based summary

In [17]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 82
Segment: seg_2 | Weight: 77
Segment: seg_3 | Weight: 74
Segment: seg_4 | Weight: 75
Segment: seg_5 | Weight: 68
Segment: seg_6 | Weight: 74
Segment: seg_7 | Weight: 69
Segment: seg_8 | Weight: 52
Segment: seg_9 | Weight: 50
Segment: seg_10 | Weight: 48


In [18]:
model_artifact.get("decile_min_thresholds")

{'1': 156,
 '2': 149,
 '3': 144,
 '4': 122,
 '5': 82,
 '6': 74,
 '7': 69,
 '8': 52,
 '9': 50,
 '10': 48}

# Scoring on Entire Dataset

In [19]:
sql_query = f"""
WITH CTE AS (
SELECT *, 
CASE WHEN seg_1 = 1 THEN 82 ELSE 0 END AS seg_1_weighted,
CASE WHEN seg_2 = 1 THEN 77 ELSE 0 END AS seg_2_weighted,
CASE WHEN seg_3 = 1 THEN 74 ELSE 0 END AS seg_3_weighted,
CASE WHEN seg_4 = 1 THEN 75 ELSE 0 END AS seg_4_weighted,
CASE WHEN seg_5 = 1 THEN 68 ELSE 0 END AS seg_5_weighted,
CASE WHEN seg_6 = 1 THEN 74 ELSE 0 END AS seg_6_weighted,
CASE WHEN seg_7 = 1 THEN 69 ELSE 0 END AS seg_7_weighted,
CASE WHEN seg_8 = 1 THEN 52 ELSE 0 END AS seg_8_weighted,
CASE WHEN seg_9 = 1 THEN 50 ELSE 0 END AS seg_9_weighted,
CASE WHEN seg_10 = 1 THEN 48 ELSE 0 END AS seg_10_weighted,
FROM pred_data),
CTE2 AS (
SELECT *, (COALESCE(seg_1_weighted,0) + COALESCE(seg_2_weighted,0) + COALESCE(seg_3_weighted,0) 
+ COALESCE(seg_4_weighted,0) + COALESCE(seg_5_weighted,0) + COALESCE(seg_6_weighted,0) + COALESCE(seg_7_weighted,0) 
+ COALESCE(seg_8_weighted,0) + COALESCE(seg_9_weighted,0) + COALESCE(seg_10_weighted,0) ) AS total_weight
FROM CTE)
SELECT *, CASE 
WHEN total_weight >= 156 THEN 1
WHEN total_weight >= 149 THEN 2
WHEN total_weight >= 144 THEN 3
WHEN total_weight >= 122 THEN 4
WHEN total_weight >= 82 THEN 5
WHEN total_weight >= 74 THEN 6
WHEN total_weight >= 69 THEN 7
WHEN total_weight >= 52 THEN 8
WHEN total_weight >= 50 THEN 9
WHEN total_weight >= 48 THEN 10
ELSE 0 END AS decile_band

FROM CTE2
"""

# execute the query and fetch the result directly as a pyarrow table
pred_data = con.execute(sql_query).arrow()
pred_data = pa.Table.from_batches(pred_data)

In [20]:
revised_target_col = target_col+'_binary'

In [21]:
decile = con.execute(f"""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM({revised_target_col}) AS events, 
                    (SUM({revised_target_col})*100.0/COUNT(*)) AS response_rate
FROM pred_data
GROUP BY decile_band
ORDER BY decile_band
""").arrow()
decile = pa.Table.from_batches(decile)


## Check the Decile bands wise response rate

In [22]:
decile.to_pandas()


,decile_band,count,events,response_rate
0,0,25769,3512,13.628779
1,1,980,811,82.755102
2,2,889,705,79.302587
3,3,428,337,78.738318
4,4,434,296,68.202765
5,5,764,517,67.670157
6,6,894,503,56.263982
7,7,512,274,53.515625
8,8,714,379,53.081232
9,9,635,286,45.039370


# Cleanup

In [23]:
con.close()
del decile
del pred_data
del mod_data
os.remove(f"{db_file}")